# `FileCallbackHandler`

Callback handler used to write chain, agent, tool, and text callback outputs into a file.

Using it as a context manager is recommended because the file is closed automatically after use.
- Bases: `BaseCallbackHandler`
## Constructor
```python
FileCallbackHandler(
    filename: str, # File path where callback output will be written
    mode: str = "a", # File mode; "a" appends and "w" overwrites
    color: str | None = None # Default output colour
)
```
## Attributes
* `filename` — Stores the path of the output file.
* `mode` — Stores the file opening mode.
* `color` — Stores the default output colour.
* `file` — Represents the opened file object used for writing.

## Methods
1. `close`: Closes the callback output file.
   * It is safe to call this method multiple times.
   * It closes the file only when it is currently open.
   - **Syntax:**
     ```python
     close(
         self
     ) -> None
     ```

2. `on_chain_start`: Writes a message when a chain starts running.
   * Useful for recording when chain execution begins.
   - **Syntax:**
     ```python
     on_chain_start(
         self,
         serialized: dict[str, Any], # Information about the chain
         inputs: dict[str, Any], # Inputs passed to the chain
         **kwargs: Any # Additional arguments such as chain name
     ) -> None
     ```

3. `on_chain_end`: Writes a message when a chain finishes successfully.
   * Useful for recording the completion of a chain.
   - **Syntax:**
     ```python
     on_chain_end(
         self,
         outputs: dict[str, Any], # Final outputs produced by the chain
         **kwargs: Any # Additional arguments
     ) -> None
     ```

4. `on_agent_action`: Writes the action log when an agent selects an action.
   * Useful for recording which tool or action the agent decided to use.
   - **Syntax:**
     ```python
     on_agent_action(
         self,
         action: AgentAction, # Agent action containing the log
         color: str | None = None, # Colour used for this output
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

5. `on_tool_end`: Writes the result returned by a tool.
   * Optional prefixes can be added before and after the tool output.
   - **Syntax:**
     ```python
     on_tool_end(
         self,
         output: str, # Output returned by the tool
         color: str | None = None, # Colour used for this output
         observation_prefix: str | None = None, # Text written before the output
         llm_prefix: str | None = None, # Text written after the output
         **kwargs: Any # Additional arguments
     ) -> None
     ```

6. `on_text`: Writes arbitrary or intermediate text into the file.
   * Useful for storing progress messages and intermediate output.
   - **Syntax:**
     ```python
     on_text(
         self,
         text: str, # Text to write into the file
         color: str | None = None, # Colour used for this output
         end: str = "", # Text appended after the output
         **kwargs: Any # Additional arguments
     ) -> None
     ```

7. `on_agent_finish`: Writes the final log when an agent finishes.
   * Useful for recording the agent's final result.
   - **Syntax:**
     ```python
     on_agent_finish(
         self,
         finish: AgentFinish, # Agent finish object containing the final log
         color: str | None = None, # Colour used for this output
         **kwargs: Any # Additional arguments
     ) -> None
     ```

In [1]:
from langchain_core.callbacks.file import FileCallbackHandler  # File handler
from langchain_core.agents import AgentAction, AgentFinish  # Agent objects

with FileCallbackHandler("callback.log", mode="w", color=None) as handler:  # Opens file
    print(handler.filename)  # filename attribute
    print(handler.mode)  # mode attribute
    print(handler.color)  # color attribute
    print(handler.file.closed)  # file attribute

    handler.on_chain_start(  # Demonstrates chain start
        {"name": "MathChain"},
        {"question": "What is 2 + 3?"}
    )

    handler.on_text("AI is thinking...\n")  # Writes intermediate text

    action = AgentAction(  # Creates agent action
        tool="Calculator",
        tool_input="2 + 3",
        log="Agent selected Calculator\n"
    )
    handler.on_agent_action(action)  # Writes agent action

    handler.on_tool_end(  # Writes tool result
        "5",
        observation_prefix="Tool result: ",
        llm_prefix="\n"
    )

    finish = AgentFinish(  # Creates final agent result
        return_values={"output": "5"},
        log="Agent final answer: 5"
    )
    handler.on_agent_finish(finish)  # Writes agent finish

    handler.on_chain_end({"answer": "5"})  # Demonstrates chain end

    handler.close()  # Closes the file
    print(handler.file.closed)  # True means file is closed

with open("callback.log") as file:  # Opens saved log
    print(file.read())  # Displays file content

callback.log
w
None
False
True


> Entering new MathChain chain...
AI is thinking...
Agent selected Calculator

Tool result: 5

Agent final answer: 5

> Finished chain.

